# Custom tools

_Give an agent typed, documented Python functions it can call._

⏱️ **~15 min** · Part of the [JAZ](https://jaz-lang.ai) tutorial series.

> Run the setup cells once, then work through the rest top to bottom.


## Setup

In [ ]:
# Install JAZ from PyPI
%pip install -q jaz-lang

In [ ]:
import os, getpass

# JAZ routes to providers through LiteLLM and reads each provider's standard
# API-key environment variable. Set at least one. In Colab you can also use the
# secrets manager (key icon in the left sidebar).
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')


In [ ]:
import jaz
from jaz.llm import LiteLLM

# Pick the model all agents use. The model id is a LiteLLM route, 'provider/model'.
# Swap to an Anthropic model (and set ANTHROPIC_API_KEY) if you prefer.
jaz.configure(llm=LiteLLM(model='openai/gpt-5-mini'))


## Define a tool

Tools are ordinary Python functions. Their signatures and docstrings are surfaced to
the agent, so it knows how and when to call them. Nothing is registered up front.


In [ ]:
from jaz import invoke
from jaz.hooks import ReturnType

def celsius_to_fahrenheit(c: float) -> float:
    """Convert a temperature in Celsius to Fahrenheit."""
    return c * 9 / 5 + 32


## Pass it as a tool

A tool is just a keyword argument. The agent sees its name, signature, and docstring,
and calls it from its own code.


In [ ]:
result = invoke(
    ReturnType(float),
    task="What is 37 degrees Celsius in Fahrenheit?",
    celsius_to_fahrenheit=celsius_to_fahrenheit,
)
result


## Group related tools

To offer several tools under one name, pass an **object**. Its public methods become a
catalog the agent calls as `name.method(...)`.


In [ ]:
class Text:
    """Small text utilities."""

    def word_count(self, s: str) -> int:
        """Count the words in a string."""
        return len(s.split())

    def shout(self, s: str) -> str:
        """Uppercase a string."""
        return s.upper()

result = invoke(
    ReturnType(str),
    task="Count the words in 'jaz reasons in code', then shout the result. "
         "Use text.word_count and text.shout.",
    text=Text(),
)
result


## Why this works

- **Scoped** — the agent only sees the tools you pass.
- **Typed & documented** — signatures and docstrings are the agent's reference.
- **Just Python** — no registration; pass a function or an object as a keyword argument.

To make a tool propagate automatically into nested `jaz.invoke` calls, bind it with
`jaz.scope(...)`.

Continue with [recursive agents & budgets](https://jaz-lang.ai/docs/tutorials).
